# Communal gardens pipeline: CSV to GeoJSON

This notebook adds a fourth layer to CatMap: the City of Montréal's **community and collective gardens** (jardins communautaires et collectifs).

```
data/communal_gardens.csv      input: the City of Montréal's open data
data/communal_gardens.geojson  output, written by this notebook
site/app.js                    draws it as orange dots (off until you tick it in the layer switcher)
```

The CSV has `longitude` and `latitude` columns in degrees, so this follows Lab 4's **Recipe 1**. Run each cell in order, top to bottom.

## 1) Import libraries and load the CSV

In [1]:
import os
import pandas as pd
import geopandas as gpd

in_path = "../data/communal_gardens.csv"
df = pd.read_csv(in_path)
print(len(df), "gardens in", df["arrondissement"].nunique(), "boroughs")
print(df.columns.tolist())

109 gardens in 16 boroughs
['id', 'nom', 'arrondissement', 'adresse', 'date_creation', 'date_revision', 'longitude', 'latitude']


## 2) Drop gardens without coordinates, make points
Points need longitude first (x), then latitude (y), in `EPSG:4326`.

In [2]:
before = len(df)
df = df.dropna(subset=["latitude", "longitude"])
print("before:", before, "after:", len(df))

gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326")
gdf.geometry.head()

before: 109 after: 109


0    POINT (-73.59089 45.55443)
1     POINT (-73.5707 45.56787)
2    POINT (-73.61239 45.52959)
3    POINT (-73.58247 45.55196)
4     POINT (-73.5707 45.56787)
Name: geometry, dtype: geometry

## 3) Keep only what the popups show, then write
Three columns with short English names. Every address ends in `, Montréal, QC, CA, <postal code>`, which is the same for all of them, so we cut the `, Montréal, QC, CA` part and keep the postal code. The `date_creation` column is when the city made the record, not when the garden opened, so we leave it out. **Output:** `../data/communal_gardens.geojson`

In [3]:
keep = gpd.GeoDataFrame({
    "name": gdf["nom"],
    "borough": gdf["arrondissement"],
    "address": gdf["adresse"].str.replace(", Montréal, QC, CA", "", regex=False),
}, geometry=gdf.geometry, crs=gdf.crs)

out_path = "../data/communal_gardens.geojson"
keep.to_file(out_path, driver="GeoJSON", COORDINATE_PRECISION=6)
print("Wrote", out_path, "-", round(os.path.getsize(out_path) / 1_000_000, 3), "MB")
keep.head()

Wrote ../data/communal_gardens.geojson - 0.028 MB


,name,borough,address,geometry
0,Jardin citoyen du parc Beaubien,Rosemont–La Petite-Patrie,Angle de la rue Saint-Zotique Est et de la 9e ...,POINT (-73.59089 45.55443)
1,Jardin citoyen du parc de la Louisiane,Rosemont–La Petite-Patrie,"Rue de Bellechasse, entre la 31e Avenue et la ...",POINT (-73.5707 45.56787)
2,Jardin collectif des AmiEs du parc des Gorilles,Rosemont–La Petite-Patrie,"Entre les rues Beaubien Ouest, Saint-Urbain, S...",POINT (-73.61239 45.52959)
3,Jardin collectif des pensées,Rosemont–La Petite-Patrie,"8e Avenue, entre la rue de Bellechasse et le b...",POINT (-73.58247 45.55196)
4,Jardin collectif du parc de la Louisiane,Rosemont–La Petite-Patrie,"Rue de Bellechasse, entre la 31e Avenue et la ...",POINT (-73.5707 45.56787)


## 4) Check the file
The check cell from Lab 4, with the four settings changed for this file.

In [4]:
# ---- settings: change these for your own data ----
FILE = "../data/communal_gardens.geojson"
WEST, EAST, SOUTH, NORTH = -73.98, -73.47, 45.40, 45.71   # a box around the island of Montréal, in degrees
EXPECTED = 109            # how many features you expect (rows after your filters)
POPUP_FIELDS = ["name", "borough", "address"]
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Point']
All checks passed


## 5) Next: open the map
1. In VS Code, right-click `site/index.html` and choose Open with Live Server.
2. The gardens start hidden: tick **Communal gardens** in the layer switcher (top right) to show them as orange dots.